please to run this project download the csv file here 

https://drive.google.com/file/d/1WByAMpHJXZ5oTY120KEvBGRxTZzXoNQC/view?usp=sharing

In [ ]:
import pandas as pd

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
from requests import api


load_dotenv (override = True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith("sk-proj-") : 
  print ("API Key looks good so far")
else : 
  print ("There might be a proble, with your API key ! ")
  

In [ ]:
Model = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
book_scraped = pd.read_csv ("books_scraped.csv")
book_scraped

In [ ]:
Book_category = book_scraped["Book_category"].drop_duplicates().tolist()
Book_category

In [ ]:
book_scraped.drop("Stock" , axis= 1 , inplace= True)
book_scraped

In [ ]:
def smart_search (store_data ,category_name = None , max_price = None ) : 
   result = store_data.copy()
   if category_name : 
      result = result[result["Book_category"].str.lower() == category_name.lower()]
   
   final_result = result [result ["Price"] <= max_price ]
   if final_result.empty and max_price : 
      suggestions= result [ result ["Price"] > max_price].sort_values(by ="Price").head(5)
      message =  f"""
      The book category you want is more expensive than the budget or doesn't exist.
      there is some suggestions of category {category_name} 
       """
      return message , suggestions 
   
   return final_result.sort_values(by ="Price").head(5)


In [ ]:
type (smart_search(book_scraped , "Novels" , 50) )

In [ ]:
system_prompt = f"""
 You are a professional and friendly AI Book Consultant for the "the Bookfather " bookstore.
 Objective: Your goal is to help customers find their perfect book based on a specific dataset.
  Workflow:

  Greeting: Start with a warm welcome.
  Information Gathering: You MUST ask the user for two things:
  Their preferred Book Category (e.g., Fiction, History, Poetry).
  Their Maximum Budget (Price).
  Guidance: Once the user provides this, inform them that you are searching the catalog.

  Tone: Helpful, concise, and professional. Do not hallucinate books that are not in the store's data
"""

In [ ]:
def return_book_info ( category , max_price ) :
  df = book_scraped.copy()
  result = smart_search (
    df,
    category_name= category ,
    max_price= max_price
  )
  if type (result) == tuple : 
    return result 
  else :
    json_str = result.to_json(orient="records", indent=2)
    return json_str




In [ ]:
def openai_stream (category , budget) : 
  books_json = return_book_info(category , budget)
  prompt = f"""
The user is looking for books in the category: '{category}' 
with a maximum budget of: {budget}.

Based on our database, here are the top 5 matches in JSON format:
{books_json}

Please introduce these books to the user, highlighting their price and rating, and tell them why these are the best choices for them.
"""
  messages = [
    {"role" : "system" , "content" : system_prompt},
    {"role" : "user" , "content" : prompt} 
  ]

  stream = openai.chat.completions.create( 
    model = Model,
    messages = messages, 
    stream = True 
  )
  result = ""
  for chunk in stream : 
      result += chunk.choices[0].delta.content or ""
      yield result

In [ ]:
print (smart_search(book_scraped ,"Novels" , 70))

In [ ]:
import gradio as gr 

In [ ]:

category_selector = gr.Dropdown(Book_category , label = "select category" , value = "Poetry")
max_price_input = gr.Number(label = "Max-value" , value = 70)
message_output = gr.Markdown (label = "Response")

view = gr.Interface (
  fn = openai_stream ,
  title = "The BookFather ",
  inputs = [category_selector , max_price_input],
  outputs = [message_output],
  flagging_mode = "never"
)

view.launch ()